# residual-stream-dynamics — Colab runner

Runs the corpus workflows on a free-tier GPU. Execute the setup cells
(0–5) in order once per session, then run whichever analysis cells you need.

**Free-tier notes**
- The GPU is a **T4 (16GB)**. `gpt2-small` through `pythia-2.8b` all fit;
  `pythia-6.9b` does **not** and is not usable here.
- The runtime is recycled after ~90 min idle / 12 h max, and `/content` is
  wiped with it. Step 4 mounts Drive so `data/` and `figures/` survive.
- Set the runtime to GPU first: **Runtime → Change runtime type → T4 GPU**.

## 0. Environment check

**This notebook only runs on Google Colab.** It drives a Colab GPU runtime:
it clones into `/content`, mounts Google Drive, and expects an NVIDIA GPU.
None of those exist on a local machine, so running it in local Jupyter or
VS Code will fail at the clone step.

To work locally, skip this notebook entirely and run the workflow scripts
from a terminal (see the README) — they already run on CPU/MPS.

In [ ]:
# Stop early with a clear message if this is not a Colab runtime.
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    raise RuntimeError(
        "This notebook only runs on Google Colab.\n\n"
        "It clones into /content, mounts Google Drive, and needs an NVIDIA GPU "
        "— none of which exist locally.\n\n"
        "To run it: open https://colab.research.google.com, then\n"
        "  File > Open notebook > GitHub > storyofthewolf/residual-stream-dynamics\n"
        "  select this notebook, then Runtime > Change runtime type > T4 GPU.\n\n"
        "To work locally instead, run the workflow scripts from a terminal:\n"
        "  python workflows/single_prompt.py --model gpt2-small\n"
        "They already run on CPU/MPS and need none of this setup."
    )

print(f"Running on Google Colab (python {sys.version.split()[0]})")

## 1. Verify the GPU is attached

In [ ]:
import shutil, torch

if shutil.which("nvidia-smi"):
    !nvidia-smi
else:
    print("nvidia-smi not found — no NVIDIA driver on this machine.")

print()
print(f"torch {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"device: {p.name}  ({p.total_memory / 1024**3:.1f} GB)")
else:
    raise RuntimeError(
        "No GPU attached.\n\n"
        "Runtime > Change runtime type > T4 GPU, then Runtime > Run all.\n"
        "Every workflow below would otherwise fall back to CPU and run "
        "many times slower."
    )

## 2. Clone the repository

`data/` and `figures/` are gitignored, so this clone is small (a few MB).
Results are written to Drive in step 4.

**Check `BRANCH`** below — it must name a branch already pushed to GitHub,
since Colab clones from the remote, not from your laptop. Set it back to
`"main"` once the work is merged. The cell prints the checked-out branch and
HEAD commit so you can confirm what is actually running.

In [2]:
import os
from pathlib import Path

REPO_URL  = "https://github.com/storyofthewolf/residual-stream-dynamics.git"
REPO_NAME = "residual-stream-dynamics"
BRANCH    = "main"   # <-- set to your working branch if not yet merged
REPO_DIR  = Path("/content") / REPO_NAME

if REPO_DIR.exists():
    print(f"{REPO_DIR} already exists — fetching {BRANCH}")
    !cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

if not REPO_DIR.exists():
    raise RuntimeError(
        f"Clone failed — {REPO_DIR} does not exist.\n\n"
        f"Most likely BRANCH ({BRANCH!r}) is not pushed to GitHub yet, or the "
        f"name is misspelled. Check with:\n"
        f"  git ls-remote --heads {REPO_URL}"
    )

os.chdir(REPO_DIR)
print(f"\ncwd: {Path.cwd()}")
!git rev-parse --abbrev-ref HEAD
!git log --oneline -1

fatal: could not create leading directories of '/content/residual-stream-dynamics': Read-only file system


FileNotFoundError: [Errno 2] No such file or directory: '/content/residual-stream-dynamics'

## 3. Install dependencies

Colab ships its own torch build — we deliberately do **not** reinstall it, since
pip would pull a CPU-only or CUDA-mismatched wheel and break GPU support.
Only the packages Colab lacks are installed.

In [ ]:
# Install everything EXCEPT torch (Colab's preinstalled build is CUDA-matched).
!pip install -q transformer_lens sae_lens

import torch
print(f"\ntorch {torch.__version__} — cuda available: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "GPU lost after install — Runtime > Restart, then rerun."

## 4. Mount Drive and redirect outputs

`data/` and `figures/` become symlinks into Drive, so every `.npz` and `.png`
the workflows write persists across runtime restarts. Nothing in the repo code
changes — the workflows still resolve paths from `_PROJECT_ROOT`, which now
points through the symlink.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, os

drive.mount("/content/drive")

# Everything lives under one Drive folder; change if you prefer another location.
DRIVE_ROOT = Path("/content/drive/MyDrive/residual-stream-dynamics")
(DRIVE_ROOT / "data").mkdir(parents=True, exist_ok=True)
(DRIVE_ROOT / "figures").mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/residual-stream-dynamics")

for name in ("data", "figures"):
    local = REPO_DIR / name
    target = DRIVE_ROOT / name
    if local.is_symlink():
        local.unlink()
    elif local.exists():
        # A fresh clone has an empty dir (just .gitkeep) — safe to replace.
        shutil.rmtree(local)
    local.symlink_to(target)
    print(f"{local}  ->  {target}")

print("\nOutputs will persist in Drive across runtime restarts.")

## 5. Smoke test

Confirms the model loads on CUDA and the pipeline runs end to end before you
commit to a long job.

In [ ]:
!python workflows/single_prompt.py --model gpt2-small --hooks resid_post \
    --logit-lens --no-plots --alpha 1.0

---
## Analysis workflows

Each cell is independent. `--device` now defaults to auto-detect, so CUDA is
picked up automatically — no flag needed.

Add `--run-tag <name>` to keep successive runs from overwriting each other.

### Entropy analysis

In [ ]:
!python workflows/entropy_analysis.py --model gpt2-small \
    --corpus corpus/base_vs_contrast_n216.json --save-data

### Ablation analysis (the primary experiment)

In [ ]:
!python workflows/ablation_analysis.py --model gpt2-small \
    --corpus corpus/base_vs_contrast_n216.json \
    --ev-thresholds 0.50 0.75 0.90 0.95 0.99 \
    --save-data

### c_k spectrum analysis

`--last-token-only` stores `[n_layers, 1, d_model]` instead of the full token
axis, shrinking the `.npz` by roughly the sequence length. Eight of the nine
c_k figures use only the final token, so this discards nothing they need; the
all-tokens heatmap is skipped automatically.

In [ ]:
!python workflows/ck_analysis.py --model gpt2-small \
    --corpus corpus/base_vs_contrast_n216.json \
    --last-token-only --save-data

### W_U subspace analysis

In [ ]:
!python workflows/wu_subspace_analysis.py --model gpt2-small \
    --corpus corpus/base_vs_contrast_n216.json --save-data

### Mechanics analysis

In [ ]:
!python workflows/mechanics_analysis.py --model gpt2-small \
    --corpus corpus/base_vs_contrast_n216.json --save-data

---
## Sweeping models

`pythia-2.8b` and `gpt2-xl` load in float16 automatically on CUDA so they fit
the T4. `pythia-6.9b` is deliberately excluded — it does not fit in 16GB.

Run this unattended; results accumulate in Drive. If the runtime dies partway,
completed models are already saved and you can restart from the survivors.

In [ ]:
MODELS = ["gpt2-small", "gpt2-medium", "gpt2-large", "pythia-160m", "pythia-1b"]

for m in MODELS:
    print(f"\n{'='*70}\n  {m}\n{'='*70}")
    !python workflows/ablation_analysis.py --model {m} \
        --corpus corpus/base_vs_contrast_n216.json \
        --ev-thresholds 0.50 0.75 0.90 0.95 0.99 --save-data --no-plots

### Larger models (float16, one at a time)

Run these individually and restart the runtime between them to release VRAM.

In [ ]:
!python workflows/ablation_analysis.py --model pythia-2.8b \
    --corpus corpus/base_vs_contrast_n216.json \
    --ev-thresholds 0.50 0.90 0.99 --save-data --no-plots

---

## Moral-polarity experiment — neutral controls

The moral corpus (`corpus/corpus_moral.json`) tests whether virtue/vice has a
common signature in the residual stream. A linear probe finds one: it
generalizes to **held-out moral foundations** at 0.950 / 0.958 / 0.983 / 0.992
for gpt2 small / medium / large / xl.

The neutral corpus (`corpus/corpus_neutral.json`) is the control for that
result. It uses the **same four frames** and the same token-matching rule, but
swaps the moral contrast for non-moral antonym axes (size, temperature, speed,
sound, age). If the moral numbers reflect moral content rather than the carrier
frames or generic semantic separability, neutral should score far lower.

It already does at the two smaller models — 0.667 (small) and 0.683 (medium)
against 0.950 / 0.958 for moral. **What is missing is a matched control at
gpt2-large and gpt2-xl**, which is exactly where the moral effect is strongest.
The cells below fill that gap.

Runtime on a T4 is roughly 10 min for large and 25-40 min for xl. Both fit in
the T4's 14.6 GB usable VRAM at float16; `--dtype` is left at its default, which
auto-selects float16 for the models flagged `large_on_16gb` in MODEL_CONFIGS.

### Neutral entropy + activations (gpt2-large, gpt2-xl)

`--save-data` is required: the probe cell below reads the saved
ActivationRecords rather than running a second forward pass.

In [ ]:
import subprocess, sys
from pathlib import Path

NEUTRAL_MODELS = ["gpt2-large", "gpt2-xl"]

# Preflight: everything the run depends on, checked before burning GPU time.
assert Path("corpus/corpus_neutral.json").exists(), (
    "corpus/corpus_neutral.json missing — the clone predates it. "
    "Re-run the clone cell (it pulls) or check `git log --oneline -1`.")
import sklearn  # noqa: F401  — probe_compute needs it; Colab preinstalls it
print("preflight OK — corpus present, sklearn importable")
!git log --oneline -1

failed = []
for m in NEUTRAL_MODELS:
    print(f"\n{'='*70}\n  NEUTRAL entropy — {m}\n{'='*70}", flush=True)
    # subprocess (not !) so a non-zero exit is caught instead of scrolling past.
    r = subprocess.run(
        [sys.executable, "workflows/entropy_analysis.py", "--model", m,
         "--corpus", "corpus/corpus_neutral.json",
         "--logit-lens", "--save-data", "--no-plots", "--run-tag", "neutral"],
        capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        failed.append(m)
        print(f"!!! {m} FAILED (exit {r.returncode})")
        print(r.stderr[-3000:])

print(f"\n{'='*70}")
for m in NEUTRAL_MODELS:
    f = Path(f"data/activation_records_{m}_corpus_neutral_resid_post_neutral.npz")
    print(f"  {m:<12} activations: {'OK ' + str(round(f.stat().st_size/1e6)) + ' MB' if f.exists() else 'MISSING'}")
if failed:
    raise RuntimeError(f"Extraction failed for: {failed}. Read the stderr above "
                       f"before running the probe cell.")

### Neutral probe (gpt2-large, gpt2-xl)

Runs from the ActivationRecords saved above — no forward passes. The number to
compare against the moral result is **`peak mean`** on the
leave-one-foundation-out line (for the neutral corpus the held-out groups are
domains: size, temp, speed, sound, age).

In [ ]:
import subprocess, sys
from pathlib import Path

failed = []
for m in NEUTRAL_MODELS:
    act = Path(f"data/activation_records_{m}_corpus_neutral_resid_post_neutral.npz")
    if not act.exists():
        print(f"SKIP {m} — {act} not found (its extraction did not finish)")
        failed.append(m)
        continue
    print(f"\n{'='*70}\n  NEUTRAL probe — {m}\n{'='*70}", flush=True)
    r = subprocess.run(
        [sys.executable, "workflows/probe_analysis.py", "--model", m,
         "--load-data", str(act), "--n-perm", "200",
         "--save-data", "--run-tag", "neutral"],
        capture_output=True, text=True)
    # Keep only the summary lines; the per-layer log is long.
    for line in r.stdout.splitlines():
        if any(k in line for k in ("layer-0", "peak mean", "Saved", "Probe:", "failed")):
            print(line)
    if r.returncode != 0:
        failed.append(m)
        print(f"!!! {m} PROBE FAILED (exit {r.returncode})")
        print(r.stderr[-3000:])

if failed:
    print(f"\nIncomplete: {failed}")

### Moral vs neutral — the comparison this was all for

Prints the leave-one-group-out peak for every probe run found in `data/probe/`.
A large moral/neutral gap at gpt2-large and gpt2-xl is what makes the headline
result defensible; a narrow one would mean larger models linearly separate any
semantic contrast and the moral finding is not specific.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / "src"))
from probe_compute import load_probe_records

probe_dir = Path("data/probe")
files = sorted(probe_dir.glob("probe_records_*.npz")) if probe_dir.exists() else []

if not files:
    # An empty table is ambiguous, so say which step did not produce output.
    print("No probe records found in data/probe/.\n")
    print(f"  data/probe exists: {probe_dir.exists()}")
    print(f"  data/ contents:")
    for f in sorted(Path("data").glob("*"))[:15]:
        print(f"    {f.name}")
    print("\nMost likely the probe cell above did not complete. Scroll up and "
          "look for '!!! ... FAILED' or a Traceback.")
else:
    rows = []
    for f in files:
        for r in load_probe_records(f):
            if r.generalize_by == "foundation":
                corpus = "moral" if "moral" in r.corpus_tag else "neutral"
                rows.append((r.model_name, corpus, r.n_layers,
                             float(r.accuracy.max()), int(r.accuracy.argmax())))

    order = {"gpt2-small": 0, "gpt2-medium": 1, "gpt2-large": 2, "gpt2-xl": 3}
    rows.sort(key=lambda x: (order.get(x[0], 9), x[1]))

    print(f"{'model':<13}{'corpus':<10}{'layers':>7}{'leave-1-out':>13}{'@layer':>8}")
    print("-" * 51)
    for m, c, nl, acc, pk in rows:
        print(f"{m:<13}{c:<10}{nl:>7}{acc:>13.3f}{pk:>8}")

    by = {}
    for m, c, nl, acc, pk in rows:
        by.setdefault(m, {})[c] = acc
    print()
    for m, d in by.items():
        if "moral" in d and "neutral" in d:
            print(f"{m:<13} moral - neutral = {d['moral'] - d['neutral']:+.3f}")
        elif "moral" in d:
            print(f"{m:<13} moral {d['moral']:.3f}   (no neutral control yet)")
        else:
            print(f"{m:<13} neutral {d['neutral']:.3f}   (no moral run on this machine)")


---
## Free VRAM between models

Colab does not release GPU memory until the process exits. The `!python`
invocations above each run in their own process, so VRAM is freed automatically
when they finish. Use this only if you loaded a model inside the notebook itself.

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM allocated: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")
print(f"VRAM reserved:  {torch.cuda.memory_reserved(0) / 1024**3:.2f} GB")

---
## Check what has been saved to Drive

In [ ]:
!du -sh /content/drive/MyDrive/residual-stream-dynamics/data/* 2>/dev/null
!echo "---"
!ls -lh /content/drive/MyDrive/residual-stream-dynamics/data/ablation/ 2>/dev/null | tail -20